In [1]:
!git clone https://github.com/Arcsin720/Sport-News.git 2>/dev/null || (cd Sport-News && git pull)
%cd /content/Sport-News

Already up to date.
/content/Sport-News


In [ ]:

# Fine-tuning mT5-small sur MLSUM-sport

import torch, gc

# Setup
%cd /content/Sport-News
print(f"GPU : {torch.cuda.get_device_name(0)}")

from datasets import load_dataset, Dataset
import pandas as pd

#  1. Dataset 
SPORT_TOPICS = [
    "athletisme", "basket", "blog-du-tour-de-france", "blog-roland-garros",
    "championnats-monde-athletisme", "coupe-du-monde", "coupe-du-monde-rugby",
    "cyclisme", "football", "formule-1", "golf", "handball",
    "jeux-olympiques", "jeux-olympiques-pyeongchang-2018", "jeux-olympiques-rio-2016",
    "le-nouveau-roland-garros", "ligue-1", "ligue-des-champions",
    "natation", "roland-garros", "rugby", "ski", "sport",
    "tennis", "top-14", "tour-de-france", "voile",
]
BASE_URL = "hf://datasets/reciTAL/mlsum@refs/convert/parquet/fr"

dataset = load_dataset(
    "parquet",
    data_files={"train": f"{BASE_URL}/train/*.parquet",
                "validation": f"{BASE_URL}/validation/*.parquet"},
)

df_train = dataset["train"].to_pandas()
df_train = df_train[df_train["topic"].isin(SPORT_TOPICS)].sample(n=5000, random_state=42).reset_index(drop=True)

df_val = dataset["validation"].to_pandas()
df_val = df_val[df_val["topic"].isin(SPORT_TOPICS)].sample(n=500, random_state=42).reset_index(drop=True)

del dataset; gc.collect()
print(f"Train : {len(df_train)} | Val : {len(df_val)}")

#2. Tokenisation
from transformers import AutoTokenizer

MODEL_NAME = "google/mt5-small"      # ← NOUVEAU MODÈLE, beaucoup plus léger
MAX_INPUT_LENGTH = 384
MAX_TARGET_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Le préfixe "summarize: "
def preprocess(batch):
    inputs = ["summarize: " + t for t in batch["text"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["summary"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(df_train[["text", "summary"]])
val_ds   = Dataset.from_pandas(df_val[["text", "summary"]])

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=["text", "summary"])
val_tokenized   = val_ds.map(preprocess, batched=True, remove_columns=["text", "summary"])

del df_train, df_val, train_ds, val_ds; gc.collect()
print(f"Tokenisation OK")

# ---------- 3. Modèle + Trainer ----------
from transformers import (
    AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
)

torch.cuda.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest")

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-sport-finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,                       # ← mt5-small
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,
    predict_with_generate=False,
    bf16=True,
    report_to="none",
    push_to_hub=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    optim="adafactor",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Trainer prêt, lancement du fine-tuning...")
print(f"VRAM utilisée avant train : {torch.cuda.memory_allocated(0)/1e9:.2f} Go")

# 4. Training 
trainer.train()

/content/Sport-News
GPU : Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train : 5000 | Val : 500


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss
500,5.210946,2.461920
1000,4.409780,2.285285
1250,4.140629,2.232284


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=6.301607153320313, metrics={'train_runtime': 1460.3199, 'train_samples_per_second': 6.848, 'train_steps_per_second': 0.856, 'total_flos': 3965622681600000.0, 'train_loss': 6.301607153320313, 'epoch': 2.0})

In [3]:
# Sauvegarde du modèle fine-tuné
SAVE_DIR = "./mt5-sport-finetuned-final"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f" Modèle sauvegardé dans {SAVE_DIR}")
!ls -lh {SAVE_DIR}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Modèle sauvegardé dans ./mt5-sport-finetuned-final
total 1.2G
-rw-r--r-- 1 root root  803 Jun  7 20:51 config.json
-rw-r--r-- 1 root root  151 Jun  7 20:51 generation_config.json
-rw-r--r-- 1 root root 1.2G Jun  7 20:51 model.safetensors
-rw-r--r-- 1 root root 2.7K Jun  7 20:52 tokenizer_config.json
-rw-r--r-- 1 root root  16M Jun  7 20:52 tokenizer.json
-rw-r--r-- 1 root root 5.3K Jun  7 20:52 training_args.bin


In [4]:

# Test qualité : vanilla vs fine-tuné
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Modèle vanilla (mt5-small de Google, non spécialisé)
tokenizer_vanilla = AutoTokenizer.from_pretrained("google/mt5-small")
model_vanilla = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small").to("cuda")
model_vanilla.eval()

# Modèle fine-tuné
tokenizer_ft = AutoTokenizer.from_pretrained(SAVE_DIR)
model_ft = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR).to("cuda")
model_ft.eval()

# Recharger 3 articles test
from datasets import load_dataset
BASE_URL = "hf://datasets/reciTAL/mlsum@refs/convert/parquet/fr"
test_ds = load_dataset("parquet", data_files={"test": f"{BASE_URL}/test/*.parquet"})
df_test = test_ds["test"].to_pandas()
df_test_sport = df_test[df_test["topic"].isin(SPORT_TOPICS)].reset_index(drop=True)
samples = df_test_sport.sample(n=3, random_state=42).reset_index(drop=True)

def summarize(text, tokenizer, model):
    inputs = tokenizer("summarize: " + text, max_length=384, truncation=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_length=64, num_beams=4, no_repeat_ngram_size=2, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Comparaison
print("-" * 80)
print("COMPARAISON VANILLA vs FINE-TUNÉ")

for i, row in samples.iterrows():
    s_van = summarize(row["text"], tokenizer_vanilla, model_vanilla)
    s_ft  = summarize(row["text"], tokenizer_ft, model_ft)

    print(f"\n[{i+1}] {row['topic']}")
    print(f"  Référence : {row['summary']}")
    print(f"  Vanilla   : {s_van}")
    print(f"  Fine-tuné : {s_ft}")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


fr/test/0000.parquet:   0%|          | 0.00/42.6M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

--------------------------------------------------------------------------------
COMPARAISON VANILLA vs FINE-TUNÉ

[1] sport
  Référence : Le Suédois a présidé pendant dix-sept ans, de 1990 à 2007, l’UEFA, avant d’être battu par Michel Platini. Il fut nommé dans la foulée président honoraire de l’UEFA.
  Vanilla   : <extra_id_0>.
  Fine-tuné : Le Suédois Lennart Johansson, ancien président de l’Union européenne du football, est mort à 89 ans.

[2] sport
  Référence : Le sélectionneur Jacques Brunel a annoncé mardi la composition de l’équipe de France qui affrontera l’Ecosse, samedi à Edimbourg. Le capitaine Guirado fait son retour.
  Vanilla   : <extra_id_0> !
  Fine-tuné : Après la victoire de l’équipe de France, le sélectionneur Jacques Brunel a dévoilé mardi le premier match de préparation des Bleus à la Coupe du monde.

[3] football
  Référence : Malgré une nette évolution, la pratique féminine du football souffre encore de stéréotypes sexistes. Organisé en France du 7 juin au 7 ju

In [6]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


HF_USER = "ArcSin720"
REPO_NAME = "mt5-sport-finetuned"

# Charger le modèle local et le push
model_ft = AutoModelForSeq2SeqLM.from_pretrained("./mt5-sport-finetuned-final")
tokenizer_ft = AutoTokenizer.from_pretrained("./mt5-sport-finetuned-final")

# Push
print("Upload du modèle en cours...")
model_ft.push_to_hub(f"{HF_USER}/{REPO_NAME}")
tokenizer_ft.push_to_hub(f"{HF_USER}/{REPO_NAME}")

print(f"\n Modèle disponible sur https://huggingface.co/{HF_USER}/{REPO_NAME}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Upload du modèle en cours...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpukuqiium/tokenizer.json:  98%|#########8| 16.1MB / 16.3MB            


✅ Modèle disponible sur https://huggingface.co/ArcSin720/mt5-sport-finetuned
